# Phase 2C - Classification ablation
### IQ-OTH/NCCD, trained and evaluated under the grouped (pseudo-patient) split

**Run on:** Colab, free T4 GPU. Budget ~40-60 min.

---

### What Phase 0 established

| configuration | accuracy | macro-F1 |
|---|---|---|
| Whole-image ViT (its own training distribution) | 0.902 | 0.805 |
| Deployed patch pipeline, any-malignant voting | 0.541 | 0.280 |

The classifier is not the weak component. Feeding it 30x30 crops it was never trained
on costs 36 accuracy points. Phase 0 also measured an **11.8 point** gap between the
image-level and grouped test splits, plus 773 near-duplicate image pairs straddling the
image-level boundary.

### What this notebook establishes

1. What a correctly-trained classifier achieves under a protocol free of the
   image-level leak.
2. Whether the **patch design is salvageable at all** when the classifier is trained on
   patches rather than on whole images. Phase 0 could not answer this: it only showed
   that a whole-image-trained model fails on patches. Training on patches is the
   experiment that decides whether the pipeline architecture is recoverable or should
   be abandoned.
3. How much of the remaining error is class imbalance (561 malignant / 416 normal /
   120 benign) rather than representation.

### Protocol, fixed before any model is trained

- The **frozen grouped split from Phase 0** is downloaded, never recomputed. Groups are
  pseudo-patients recovered by similarity clustering (117 groups against 110 known
  cases).
- Model selection uses **validation only**. Test is touched once, at the end, for the
  final table.
- **Balanced accuracy and macro-F1 are the headline metrics.** Plain accuracy is
  misleading here: predicting "malignant" for everything scores 0.51.
- The grouping is approximate. Benign is over-split (29 groups against 15 known cases),
  which leaves residual same-patient leakage, so these numbers are best read as an
  **upper bound** on true patient-level performance.

## 1. Environment and data

In [ ]:
import subprocess, sys
for pkg in ["opencv-python-headless", "tabulate", "kagglehub"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg], check=False)

import tensorflow as tf, keras
gpus = tf.config.list_physical_devices("GPU")
print("tensorflow:", tf.__version__, "| keras:", keras.__version__)
print("GPUs      :", gpus)
assert gpus, "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."

In [ ]:
import os, json, time, shutil, random
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (confusion_matrix, precision_recall_fscore_support,
                             accuracy_score, balanced_accuracy_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

CLASS_NAMES = ["Benign", "Malignant", "Normal"]
FOLDERS = {"Benign": "Bengin cases", "Malignant": "Malignant cases",
           "Normal": "Normal cases"}
LABEL_TO_IDX = {"Benign": 0, "Malignant": 1, "Normal": 2}

RESULTS_DIR = "/content/fyp_phase2c_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = {"seed": SEED}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)
print("results ->", RESULTS_DIR)

### 1.1 Kaggle credentials

`KAGGLE_API_TOKEN` must exist in Colab Secrets **with notebook access enabled for this
notebook**. The access toggle is per notebook and resets whenever a notebook is opened
fresh from GitHub, which is the usual cause of a failure here.

In [ ]:
def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        return (v or "").strip() or None
    except Exception as e:
        if "NotebookAccess" in type(e).__name__:
            print(f"  {name}: exists, but this notebook is not granted access.")
            print("    FIX: Secrets panel -> toggle 'Notebook access' ON -> re-run.")
        elif "SecretNotFound" in type(e).__name__:
            print(f"  {name}: not present in Colab Secrets.")
        return None

if _secret("KAGGLE_API_TOKEN"):
    os.environ["KAGGLE_API_TOKEN"] = _secret("KAGGLE_API_TOKEN")
    print("OK - using KAGGLE_API_TOKEN")
elif _secret("KAGGLE_USERNAME") and _secret("KAGGLE_KEY"):
    os.environ["KAGGLE_USERNAME"] = _secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _secret("KAGGLE_KEY")
    print("OK - using legacy credentials")
else:
    raise RuntimeError("No Kaggle credential available; fix the Secrets toggle above.")

import kagglehub
DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
cands = [d for d, _, _ in os.walk(DL) if os.path.basename(d) == "Malignant cases"]
DATA_ROOT = os.path.dirname(cands[0])
print("DATA_ROOT:", DATA_ROOT)

### 1.2 The frozen split

Downloaded, not recomputed. Recomputing the similarity clustering in each notebook
would risk silently different groups and make the ablation table incomparable across
runs.

In [ ]:
SPLIT_URL = ("https://raw.githubusercontent.com/haseebkhan9081/"
             "iqothnccd-leakage-audit/main/split_seed42.csv")
subprocess.run(["wget", "-q", "-O", "/content/split_seed42.csv", SPLIT_URL], check=True)
split = pd.read_csv("/content/split_seed42.csv")

split["path"] = [os.path.join(DATA_ROOT, FOLDERS[l], f)
                 for l, f in zip(split["label"], split["file"])]
missing = [p for p in split["path"] if not os.path.exists(p)]
assert not missing, f"{len(missing)} files in the split are absent from the download"

print("images:", len(split), "| groups:", split["group"].nunique())
print(pd.crosstab(split["split_grouped"], split["label"]).to_string())
print("\ngroups per split:", split.groupby("split_grouped")["group"].nunique().to_dict())
assert split.groupby("group")["split_grouped"].nunique().max() == 1
print("verified: no group spans two splits")

RESULTS["split"] = {
    "source": "frozen from Phase 0",
    "counts": pd.crosstab(split["split_grouped"], split["label"]).to_dict(),
    "n_groups": int(split["group"].nunique())}
save_json()

## 2. Shared evaluation code

In [ ]:
def evaluate(y_true, y_pred, title):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    out = {"title": title, "n": int(len(y_true)),
           "accuracy": float(accuracy_score(y_true, y_pred)),
           "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
           "macro_f1": float(f1.mean()),
           "per_class": {c: {"precision": float(p[i]), "recall": float(r[i]),
                             "f1": float(f1[i]), "support": int(sup[i])}
                         for i, c in enumerate(CLASS_NAMES)},
           "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist(),
           "pred_distribution": {c: int((y_pred == i).sum())
                                 for i, c in enumerate(CLASS_NAMES)}}
    print(f"\n--- {title}  (n={out['n']}) ---")
    print(f"{'class':<11}{'prec':>8}{'rec':>8}{'f1':>8}{'n':>7}")
    for c in CLASS_NAMES:
        m = out["per_class"][c]
        print(f"{c:<11}{m['precision']:>8.3f}{m['recall']:>8.3f}{m['f1']:>8.3f}{m['support']:>7d}")
    print(f"acc {out['accuracy']:.3f} | BALANCED-ACC {out['balanced_accuracy']:.3f} "
          f"| MACRO-F1 {out['macro_f1']:.3f}")
    return out

def load_images(paths, size, rgb=True):
    X = np.empty((len(paths), size, size, 3), np.float32)
    for i, p in enumerate(paths):
        im = Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR)
        X[i] = np.asarray(im, np.float32)
    return X

def subset(name):
    s = split[split["split_grouped"] == name]
    return s["path"].tolist(), s["y"].to_numpy()

tr_paths, ytr = subset("train")
va_paths, yva = subset("val")
te_paths, yte = subset("test")
print("train/val/test:", len(ytr), len(yva), len(yte))

# Inverse-frequency class weights. Benign is 11% of the data and was the class the
# original pipeline missed almost entirely (recall 0.13), so this is load-bearing.
counts = np.bincount(ytr, minlength=3)
CLASS_WEIGHT = {i: float(len(ytr) / (3 * c)) if c else 0.0 for i, c in enumerate(counts)}
print("train class counts:", dict(zip(CLASS_NAMES, counts.tolist())))
print("class weights     :", {CLASS_NAMES[i]: round(w, 3) for i, w in CLASS_WEIGHT.items()})

## 3. Models

Two families:

- **ViT from scratch**, the FYP's own architecture, parameterised by input size. Row E1
  reproduces the original 30x30 design under the corrected protocol so the ablation has
  a like-for-like starting point.
- **Transfer learning** from an ImageNet-pretrained backbone. 767 training images is far
  too few to train a transformer from scratch, and this is the standard remedy.

In [ ]:
def build_vit(size, n_classes=3, patch=6, proj=64, heads=8, depth=8, mlp_head=(2048, 1024)):
    resize_to = 72 if size <= 72 else size
    num_patches = (resize_to // patch) ** 2

    class Patches(layers.Layer):
        def __init__(self, patch_size, **kw):
            super().__init__(**kw); self.patch_size = patch_size
        def call(self, images):
            b = tf.shape(images)[0]
            p = tf.image.extract_patches(
                images=images,
                sizes=[1, self.patch_size, self.patch_size, 1],
                strides=[1, self.patch_size, self.patch_size, 1],
                rates=[1, 1, 1, 1], padding="VALID")
            return tf.reshape(p, [b, -1, self.patch_size * self.patch_size * images.shape[-1]])

    class PatchEncoder(layers.Layer):
        def __init__(self, n, d, **kw):
            super().__init__(**kw)
            self.n = n
            self.proj = layers.Dense(d)
            self.pos = layers.Embedding(input_dim=n, output_dim=d)
        def call(self, x):
            return self.proj(x) + self.pos(tf.range(self.n))

    def mlp(x, units, rate):
        for u in units:
            x = layers.Dropout(rate)(layers.Dense(u, activation=tf.nn.gelu)(x))
        return x

    inp = layers.Input((size, size, 3))
    aug = keras.Sequential([
        layers.Rescaling(1.0 / 255.0),
        layers.Resizing(resize_to, resize_to),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.15, 0.15),
    ], name="aug")(inp)

    x = Patches(patch)(aug)
    enc = PatchEncoder(num_patches, proj)(x)
    for _ in range(depth):
        a = layers.LayerNormalization(epsilon=1e-6)(enc)
        a = layers.MultiHeadAttention(num_heads=heads, key_dim=proj, dropout=0.1)(a, a)
        b = layers.Add()([a, enc])
        c = layers.LayerNormalization(epsilon=1e-6)(b)
        c = mlp(c, [proj * 2, proj], 0.1)
        enc = layers.Add()([c, b])
    r = layers.Flatten()(layers.LayerNormalization(epsilon=1e-6)(enc))
    r = layers.Dropout(0.5)(r)
    r = mlp(r, list(mlp_head), 0.5)
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax")(r))


def build_transfer(size, backbone="efficientnet", n_classes=3, trainable_top=True):
    inp = layers.Input((size, size, 3))
    x = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.15, 0.15),
    ], name="aug")(inp)

    if backbone == "efficientnet":
        base = keras.applications.EfficientNetB0(
            include_top=False, weights="imagenet", input_shape=(size, size, 3))
        x = keras.applications.efficientnet.preprocess_input(x)
    else:
        base = keras.applications.ResNet50(
            include_top=False, weights="imagenet", input_shape=(size, size, 3))
        x = keras.applications.resnet50.preprocess_input(x)

    base.trainable = trainable_top
    if trainable_top:                       # freeze all but the last two blocks
        for layer in base.layers[:-20]:
            layer.trainable = False

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax")(x))

In [ ]:
def run_experiment(cfg, Xtr, Xva, ytr_, yva_, verbose=0):
    keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = cfg["build"]()
    model.compile(optimizer=keras.optimizers.Adam(cfg.get("lr", 1e-3)),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=cfg.get("patience", 12),
                                         restore_best_weights=True, verbose=0),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                             patience=5, min_lr=1e-6, verbose=0)]
    t0 = time.time()
    hist = model.fit(Xtr, ytr_, validation_data=(Xva, yva_),
                     epochs=cfg.get("epochs", 60), batch_size=cfg.get("batch", 32),
                     class_weight=CLASS_WEIGHT if cfg.get("class_weight") else None,
                     callbacks=cbs, verbose=verbose)
    # Persist weights rather than holding the model object across iterations:
    # clear_session() at the top of the next call can invalidate a retained graph,
    # which would silently corrupt the final test evaluation.
    wpath = f"{RESULTS_DIR}/{cfg['tag']}.weights.h5"
    model.save_weights(wpath)
    return model, {"minutes": (time.time() - t0) / 60,
                   "epochs_run": len(hist.history["loss"]), "weights": wpath}

## 4. Part A - whole-image classifiers

Each row changes one thing from the row above it.

| | change | tests |
|---|---|---|
| E1 | ViT from scratch, 30x30 (the FYP's design) | the original model under a correct protocol |
| E2 | ViT from scratch, 128x128 | is 30x30 resolution the binding constraint? |
| E3 | E2 + inverse-frequency class weights | is the residual error class imbalance? |
| E4 | EfficientNetB0 transfer, 224x224 | does pretraining beat from-scratch on 767 images? |
| E5 | E4 + class weights | the two fixes combined |

In [ ]:
CACHE = {}
def get_arrays(size):
    if size not in CACHE:
        CACHE[size] = (load_images(tr_paths, size), load_images(va_paths, size),
                       load_images(te_paths, size))
    return CACHE[size]

EXPERIMENTS = [
    {"tag": "E1", "name": "ViT scratch 30px (original design)", "size": 30,
     "build": lambda: build_vit(30), "epochs": 120, "batch": 32, "class_weight": False},
    {"tag": "E2", "name": "ViT scratch 128px", "size": 128,
     "build": lambda: build_vit(128, patch=16), "epochs": 80, "batch": 32,
     "class_weight": False},
    {"tag": "E3", "name": "ViT scratch 128px + class weights", "size": 128,
     "build": lambda: build_vit(128, patch=16), "epochs": 80, "batch": 32,
     "class_weight": True},
    {"tag": "E4", "name": "EfficientNetB0 transfer 224px", "size": 224,
     "build": lambda: build_transfer(224), "epochs": 40, "batch": 16, "lr": 1e-4,
     "class_weight": False},
    {"tag": "E5", "name": "EfficientNetB0 transfer 224px + class weights", "size": 224,
     "build": lambda: build_transfer(224), "epochs": 40, "batch": 16, "lr": 1e-4,
     "class_weight": True},
]

part_a, configs = {}, {}
for cfg in EXPERIMENTS:
    print(f"\n{'='*70}\n{cfg['tag']}  {cfg['name']}\n{'='*70}")
    Xtr, Xva, Xte = get_arrays(cfg["size"])
    model, meta = run_experiment(cfg, Xtr, Xva, ytr, yva)
    # score on validation immediately, while this model is still the live graph
    val = evaluate(yva, model.predict(Xva, verbose=0).argmax(1), f"{cfg['tag']} VAL")
    part_a[cfg["tag"]] = {"name": cfg["name"], "val": val, **meta}
    configs[cfg["tag"]] = cfg
    del model
    print(f"({meta['epochs_run']} epochs, {meta['minutes']:.1f} min)")

RESULTS["part_A_val"] = part_a
save_json()

In [ ]:
sel = pd.DataFrame([{"tag": k, "model": v["name"],
                     "val_bal_acc": v["val"]["balanced_accuracy"],
                     "val_macro_f1": v["val"]["macro_f1"],
                     "val_acc": v["val"]["accuracy"]}
                    for k, v in part_a.items()]).round(3)
print(sel.to_string(index=False))

BEST = sel.sort_values("val_macro_f1", ascending=False).iloc[0]["tag"]
print(f"\nselected on VALIDATION macro-F1: {BEST}  ({part_a[BEST]['name']})")
RESULTS["selected_model"] = {"tag": BEST, "name": part_a[BEST]["name"],
                             "criterion": "validation macro-F1"}
save_json()

## 5. Part B - is the patch pipeline salvageable?

Phase 0 showed a whole-image-trained ViT collapses on patches. It could not show
whether a classifier **trained on patches** works, and that is the question that decides
whether the segment-crop-classify architecture is recoverable or should be dropped.

The U-Net generates patches once for every image. Each patch inherits its parent
image's label, which is a genuine weakness of the design worth stating plainly: a
tumour-free patch cropped from a malignant scan is labelled malignant. That noise is
intrinsic to patch-level supervision without per-patch annotations, and it is part of
what is being tested here.

In [ ]:
UNET_URL = ("https://github.com/haseebkhan9081/iqothnccd-leakage-audit/"
            "releases/download/weights-v1/UNet_best_Model_checkpoint.h5")
UNET = "/content/UNet_best_Model_checkpoint.h5"
if not os.path.exists(UNET):
    subprocess.run(["wget", "-q", "-O", UNET, UNET_URL], check=True)

from tensorflow.keras import backend as K
from skimage import measure

def _dc(yt, yp):
    f1_, f2_ = K.flatten(yt), K.flatten(yp)
    i = K.sum(f1_ * f2_)
    return (2. * i + 1) / (K.sum(f1_) + K.sum(f2_) + 1)

unet = tf.keras.models.load_model(
    UNET, custom_objects={"dice_coef": _dc, "dice_coef_loss": lambda a, b: -_dc(a, b)},
    compile=False)

def preprocess_for_unet(img_bgr):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (512, 512))
    g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g.astype(np.uint8))
    _, roi = cv2.threshold(g, 127, 255, cv2.THRESH_BINARY_INV)
    roi = cv2.erode(roi, np.ones([4, 4], np.uint8))
    roi = cv2.dilate(roi, np.ones([13, 13], np.uint8))
    return ((cv2.bitwise_and(g, g, mask=roi) - 127.0) / 127.0).astype(np.float32)

PATCH = 64          # 30 was the original; 64 gives the classifier something to work with

def patches_for(img_rgb_512, mask):
    m = cv2.dilate(mask, np.ones((5, 5), np.uint8))
    out = []
    for r in measure.regionprops(measure.label(m)):
        y1, x1, y2, x2 = r.bbox
        cy, cx = (y1 + y2) // 2, (x1 + x2) // 2
        h = PATCH // 2
        y1, y2 = max(cy - h, 0), min(cy + h, 512)
        x1, x2 = max(cx - h, 0), min(cx + h, 512)
        crop = img_rgb_512[y1:y2, x1:x2]
        if crop.size:
            out.append(cv2.resize(crop, (PATCH, PATCH)))
    return out

In [ ]:
PX, PY, POWNER = [], [], []
BATCH_IMG = 16
all_paths = split["path"].tolist()
all_y = split["y"].to_numpy()

for s in tqdm(range(0, len(all_paths), BATCH_IMG), desc="extracting patches"):
    chunk = all_paths[s:s + BATCH_IMG]
    rgb, pre = [], []
    for p in chunk:
        im = cv2.imread(p)
        rgb.append(cv2.cvtColor(cv2.resize(im, (512, 512)), cv2.COLOR_BGR2RGB))
        pre.append(preprocess_for_unet(im))
    masks = (np.squeeze(unet.predict(np.stack(pre)[..., None], verbose=0), -1)
             >= 0.5).astype(np.uint8) * 255
    for k in range(len(chunk)):
        for pt in patches_for(rgb[k], masks[k]):
            PX.append(pt); PY.append(all_y[s + k]); POWNER.append(s + k)

PX = np.asarray(PX, np.float32); PY = np.asarray(PY); POWNER = np.asarray(POWNER)
owner_split = split["split_grouped"].to_numpy()[POWNER]
print("patches:", PX.shape, "| per image:", round(len(PX) / len(all_paths), 1))
print("patch class balance:", dict(zip(CLASS_NAMES, np.bincount(PY, minlength=3).tolist())))
RESULTS["part_B_patches"] = {"n_patches": int(len(PX)), "patch_px": PATCH,
                             "per_image": len(PX) / len(all_paths)}
save_json()

In [ ]:
ptr, pva, pte = (owner_split == "train"), (owner_split == "val"), (owner_split == "test")

keras.backend.clear_session(); tf.random.set_seed(SEED)
pmodel = build_transfer(PATCH)
pmodel.compile(optimizer=keras.optimizers.Adam(1e-4),
               loss="sparse_categorical_crossentropy", metrics=["accuracy"])
pmodel.fit(PX[ptr], PY[ptr], validation_data=(PX[pva], PY[pva]),
           epochs=25, batch_size=64, class_weight=CLASS_WEIGHT,
           callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=6,
                                                    restore_best_weights=True)],
           verbose=1)

patch_probs = pmodel.predict(PX, batch_size=256, verbose=0)
patch_acc = float((patch_probs.argmax(1) == PY)[pte].mean())
print(f"\npatch-level accuracy on test patches: {patch_acc:.3f}")

# image-level: mean probability over an image's patches (Phase 0 showed any-malignant
# voting is the worse rule, so it is not carried forward)
img_pred = np.full(len(all_paths), -1)
for i in range(len(all_paths)):
    sel_i = POWNER == i
    if sel_i.any():
        img_pred[i] = patch_probs[sel_i].mean(0).argmax()
covered = img_pred >= 0
te_img = (split["split_grouped"].to_numpy() == "test") & covered

res_patch = evaluate(all_y[te_img], img_pred[te_img],
                     "E6 patch-trained pipeline, mean-prob aggregation - TEST")
print(f"\nimages with no patches (excluded): {int((~covered).sum())}")
RESULTS["part_B_result"] = {"patch_level_accuracy": patch_acc, "image_level": res_patch,
                            "images_without_patches": int((~covered).sum())}
save_json()

## 6. Part C - final test evaluation

The selected whole-image model and the patch-trained pipeline, both scored once on the
held-out grouped test set. Nothing after this point informs model selection.

In [ ]:
best_cfg = configs[BEST]
keras.backend.clear_session()
best_model = best_cfg["build"]()                    # rebuilt, then weights reloaded
best_model.load_weights(part_a[BEST]["weights"])
_, _, Xte_best = get_arrays(best_cfg["size"])
res_best = evaluate(yte, best_model.predict(Xte_best, verbose=0).argmax(1),
                    f"{BEST} {part_a[BEST]['name']} - TEST")

cm = np.array(res_best["confusion_matrix"])
plt.figure(figsize=(5.2, 4.2))
plt.imshow(cm, cmap="Blues")
for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.xticks(range(3), CLASS_NAMES, rotation=20); plt.yticks(range(3), CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"{BEST} test confusion matrix")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/cm_best_test.png", dpi=150); plt.show()

shutil.copy(part_a[BEST]["weights"], f"{RESULTS_DIR}/best_classifier.weights.h5")
RESULTS["final_test"] = {"whole_image_best": res_best}
save_json()

In [ ]:
P0 = {"deployed_pipeline_grouped_test": {"accuracy": 0.508, "macro_f1": 0.291},
      "phase0_whole_image_grouped_test": {"accuracy": 0.803, "macro_f1": 0.689}}

tbl = pd.DataFrame([
    {"row": "A0", "configuration": "Original weights, deployed patch pipeline (Phase 0)",
     "accuracy": P0["deployed_pipeline_grouped_test"]["accuracy"],
     "balanced_acc": None, "macro_f1": P0["deployed_pipeline_grouped_test"]["macro_f1"]},
    {"row": "A1", "configuration": "Original weights, whole image (Phase 0)",
     "accuracy": P0["phase0_whole_image_grouped_test"]["accuracy"],
     "balanced_acc": None, "macro_f1": P0["phase0_whole_image_grouped_test"]["macro_f1"]},
    {"row": BEST, "configuration": f"Retrained: {part_a[BEST]['name']}",
     "accuracy": res_best["accuracy"], "balanced_acc": res_best["balanced_accuracy"],
     "macro_f1": res_best["macro_f1"]},
    {"row": "E6", "configuration": f"Retrained patch pipeline ({PATCH}px patches)",
     "accuracy": res_patch["accuracy"], "balanced_acc": res_patch["balanced_accuracy"],
     "macro_f1": res_patch["macro_f1"]},
]).round(3)

print("ALL ROWS ON THE GROUPED TEST SPLIT\n")
print(tbl.to_string(index=False))
tbl.to_csv(f"{RESULTS_DIR}/ablation_table.csv", index=False)
with open(f"{RESULTS_DIR}/ablation_table.md", "w") as f:
    f.write(tbl.to_markdown(index=False))
RESULTS["ablation_table"] = tbl.to_dict("records")
save_json()

shutil.make_archive("/content/fyp_phase2c_results", "zip", RESULTS_DIR)
try:
    from google.colab import files
    files.download("/content/fyp_phase2c_results.zip")
except Exception as e:
    print("download manually from the file browser:", e)

## 7. Output and validity checks

`fyp_phase2c_results.zip` contains `results.json`, `ablation_table.md`, the selected
model's weights, and the test confusion matrix.

Conditions that invalidate the run:

- **Model selection must reference validation only.** `selected_model.criterion` records
  which metric chose the row; test is read once, in section 6.
- **Balanced accuracy must be reported alongside accuracy.** Predicting malignant for
  every image scores ~0.51 accuracy on this class distribution, so accuracy alone can
  make a degenerate model look competent.
- **E6 must be compared against E1-E5 on identical test images.** Images from which the
  U-Net extracts no patches are excluded from E6 and the count is reported; if that
  count is large, E6 is scored on an easier subset and is not directly comparable.

Residual limitation: the grouped split rests on recovered pseudo-patients, not true
patient identifiers. Benign is over-split (29 groups against 15 known cases), so some
same-patient leakage remains and these figures are an upper bound on true patient-level
performance.